# Assignment 01 - POS Extraction from a News Article

**Author:** Vishal Sigdel

**Goal:** tag every word in an English news article with its part of speech,
then export only the **nouns** and **verbs** to a CSV file.

**Pipeline:** read text -> normalise characters -> tokenize -> POS tag -> filter -> export

| File | Role |
| --- | --- |
| `news.txt` | source article (plain text) |
| `VishalSigdel_POS_01.ipynb` | this notebook |
| `VishalSigdel_POS_01.csv` | output: `Word`, `POS_Tag` |

## 1. Imports and NLTK resources

NLTK ships the models separately from the library, so they are downloaded once
into `~/nltk_data`. Re-running this cell is safe - already-present packages are skipped.

- `punkt` / `punkt_tab` - sentence & word tokenizer models
- `averaged_perceptron_tagger*` - the POS tagging model

In [1]:
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize

# Tokenizer models
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# POS tagging models
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

True

## 2. Load the article

In [2]:
ARTICLE_PATH = "news.txt"

with open(ARTICLE_PATH, "r", encoding="utf-8") as file:
    news = file.read()

print(f"Characters read: {len(news)}\n")
print(news[:200])

Characters read: 7817

Broad Peak avalanche wipes out a generation of Nepali climbing greats

The deaths of Nirmal Purja and five other mountain guides on Pakistan’s Broad Peak have dealt a severe blow to Nepal’s high-altit


## 3. Normalise the text

News sites use "smart" typography. Left as-is, the tokenizer treats `don’t` and
`don't` as different tokens and accented characters split unexpectedly, so tags
get noisy. Two cleanup steps:

1. **Accent folding** - `NFKD` splits `é` into `e` + combining accent; dropping the
   combining marks leaves plain ASCII letters.
2. **Punctuation mapping** - curly quotes and dashes to their ASCII equivalents.

In [3]:
import unicodedata

# Curly punctuation -> ASCII equivalent
PUNCTUATION_MAP = {
    "\u2019": "'",   # right single quote
    "\u2018": "'",   # left single quote
    "\u201c": '"',   # left double quote
    "\u201d": '"',   # right double quote
    "\u2013": "-",   # en dash
    "\u2014": "-",   # em dash
    "\u00a0": " ",   # non-breaking space
}


def remove_accents(text):
    """Return `text` with accents stripped (cafe\u0301 -> cafe)."""
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(c for c in decomposed if not unicodedata.combining(c))


def normalise(text):
    """Fold accents and replace curly punctuation with ASCII."""
    text = remove_accents(text)
    for original, replacement in PUNCTUATION_MAP.items():
        text = text.replace(original, replacement)
    return text


news = normalise(news)
print(news[:200])

Broad Peak avalanche wipes out a generation of Nepali climbing greats

The deaths of Nirmal Purja and five other mountain guides on Pakistan's Broad Peak have dealt a severe blow to Nepal's high-altit


## 4. Tokenize into words

`word_tokenize` splits on whitespace *and* linguistic boundaries: punctuation
becomes its own token and contractions split (`don't` -> `do`, `n't`). The tagger
expects this token list as input.

In [4]:
words = word_tokenize(news)

print("Total tokens:", len(words))
print(words[:25])

Total tokens: 1372
['Broad', 'Peak', 'avalanche', 'wipes', 'out', 'a', 'generation', 'of', 'Nepali', 'climbing', 'greats', 'The', 'deaths', 'of', 'Nirmal', 'Purja', 'and', 'five', 'other', 'mountain', 'guides', 'on', 'Pakistan', "'s", 'Broad']


## 5. POS tagging

`nltk.pos_tag` returns `(word, tag)` pairs.
Tags are context-sensitive - the same spelling can tag differently
(`climbing` is `VBG` in "climbing greats", `NN` elsewhere).

In [5]:
pos_tags = nltk.pos_tag(words)

print(pos_tags[:15])

[('Broad', 'NNP'), ('Peak', 'NNP'), ('avalanche', 'NN'), ('wipes', 'VBZ'), ('out', 'RP'), ('a', 'DT'), ('generation', 'NN'), ('of', 'IN'), ('Nepali', 'NNP'), ('climbing', 'VBG'), ('greats', 'NNS'), ('The', 'DT'), ('deaths', 'NNS'), ('of', 'IN'), ('Nirmal', 'NNP')]


## 6. Keep only nouns and verbs

Penn Treebank splits each class by inflection, so we match against explicit tag
sets rather than a prefix check - clearer to read and easy to extend.

| Noun tags | Meaning | Verb tags | Meaning |
| --- | --- | --- | --- |
| `NN` | singular common | `VB` | base form |
| `NNS` | plural common | `VBD` | past tense |
| `NNP` | singular proper | `VBG` | gerund / present participle |
| `NNPS` | plural proper | `VBN` | past participle |
| | | `VBP` | present, non-3rd person |
| | | `VBZ` | present, 3rd person singular |

In [6]:
NOUN_TAGS = {"NN", "NNS", "NNP", "NNPS"}
VERB_TAGS = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"}


def classify(tag):
    """Map a Penn Treebank tag to "Noun"/"Verb", or None if neither."""
    if tag in NOUN_TAGS:
        return "Noun"
    if tag in VERB_TAGS:
        return "Verb"
    return None


extracted_words = [
    [word, classify(tag)]
    for word, tag in pos_tags
    if classify(tag) is not None
]

print("Nouns and verbs found:", len(extracted_words))
print(extracted_words[:20])

Nouns and verbs found: 633
[['Broad', 'Noun'], ['Peak', 'Noun'], ['avalanche', 'Noun'], ['wipes', 'Verb'], ['generation', 'Noun'], ['Nepali', 'Noun'], ['climbing', 'Verb'], ['greats', 'Noun'], ['deaths', 'Noun'], ['Nirmal', 'Noun'], ['Purja', 'Noun'], ['mountain', 'Noun'], ['guides', 'Noun'], ['Pakistan', 'Noun'], ['Broad', 'Noun'], ['Peak', 'Noun'], ['have', 'Verb'], ['dealt', 'Verb'], ['blow', 'Noun'], ['Nepal', 'Noun']]


## 7. Build the DataFrame and export

Columns are named `Word` and `POS_Tag` as the assignment requires, and
`index=False` keeps pandas' row numbers out of the CSV.

In [7]:
df = pd.DataFrame(extracted_words, columns=["Word", "POS_Tag"])

df.head()

,Word,POS_Tag
0,Broad,Noun
1,Peak,Noun
2,avalanche,Noun
3,wipes,Verb
4,generation,Noun


In [8]:
OUTPUT_PATH = "VishalSigdel_POS_01.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")

Saved 633 rows to VishalSigdel_POS_01.csv


## 8. Summary

In [9]:
print("Total nouns and verbs extracted:", len(df))
print()
print(df["POS_Tag"].value_counts())

Total nouns and verbs extracted: 633

POS_Tag
Noun    457
Verb    176
Name: count, dtype: int64
